# EXP002 — Domain Classifier: Keyword vs. Claude API

**Question:** Does calling the Claude API for node classification produce meaningfully better domain tags than keyword matching alone?
**Hypothesis:** Claude few-shot will outperform keyword-only by ≥5 F1 points; cost_per_node determines whether it's worth the API spend.
**Success criterion:** claude_zero_shot F1 ≥ keyword_only F1 + 0.05; otherwise keyword_only wins.
**Production impact:** `netweave/src/taxonomy.py` → `classify_node()` function.
**Author:** Chuck Wiley  **Date:** 2026-06-23

In [ ]:
import sys
sys.path.insert(0, ".")

import json
import os
import mlflow
import mlflow.tracking
import networkx as nx
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import anthropic
from sklearn.metrics import f1_score, precision_score, recall_score
from lib.niw_graph import load_graph
from lib.niw_mlflow import start_run, log_result, compare_runs

mlflow.set_tracking_uri("sqlite:///mlflow.db")

EXPERIMENT_ID = "EXP002"
EXPERIMENT_NAME = "Domain Classifier"
mlflow.set_experiment(EXPERIMENT_NAME)

# Requires ANTHROPIC_API_KEY in environment
client = anthropic.Anthropic()

In [ ]:
# Always load from shared/ — never modify source data
G = load_graph(
    nodes_path="shared/nodes.parquet",
    edges_path="shared/edges.parquet"
)
print(f"Graph loaded: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# Load hand-labeled ground truth (must be created before running)
GT_PATH = "experiments/EXP002_domain_classifier/ground_truth.json"
if not os.path.exists(GT_PATH):
    raise FileNotFoundError(
        f"{GT_PATH} not found. "
        "Hand-label 30 nodes (5 per domain) and save as JSON before running. "
        "Format: [{\"node_id\": \"n001\", \"domain_tags\": [\"biotech\"]}]"
    )

with open(GT_PATH) as f:
    ground_truth = json.load(f)

print(f"Ground truth loaded: {len(ground_truth)} labeled nodes")

In [ ]:
DOMAIN_TAGS = ["biotech", "medical_diagnostics", "investor", "software", "fintech", "other"]

FEW_SHOT_EXAMPLES = [
    {"company": "Illumina", "position": "VP Clinical Genomics", "tags": ["biotech", "medical_diagnostics"]},
    {"company": "a16z", "position": "General Partner", "tags": ["investor"]},
    {"company": "Epic Systems", "position": "Integration Engineer", "tags": ["software", "medical_diagnostics"]},
    {"company": "Novo Holdings", "position": "Investment Associate", "tags": ["investor", "biotech"]},
    {"company": "Stripe", "position": "Head of Healthcare Payments", "tags": ["fintech", "medical_diagnostics"]},
    {"company": "Calico Labs", "position": "Research Scientist", "tags": ["biotech"]},
]

def classify_keyword_only(node_data: dict) -> list:
    """Simple keyword-based classification baseline."""
    text = f"{node_data.get('employer', '')} {node_data.get('position', '')}".lower()
    tags = []
    if any(k in text for k in ["bio", "pharma", "genomic", "therapeutics", "crispr"]):
        tags.append("biotech")
    if any(k in text for k in ["clinical", "diagnostic", "hospital", "medical", "health"]):
        tags.append("medical_diagnostics")
    if any(k in text for k in ["invest", "venture", "capital", "partner", "fund"]):
        tags.append("investor")
    if any(k in text for k in ["software", "engineer", "developer", "tech", "saas"]):
        tags.append("software")
    if any(k in text for k in ["fintech", "payment", "banking", "finance", "stripe"]):
        tags.append("fintech")
    return tags or ["other"]

def classify_claude(node_data: dict, few_shot: bool = False, with_brave: bool = False) -> tuple:
    """Classify via Claude API. Returns (tags, cost_usd)."""
    examples_text = ""
    if few_shot:
        examples_text = "\n".join([
            f"Company: {e['company']}, Position: {e['position']} → Tags: {e['tags']}"
            for e in FEW_SHOT_EXAMPLES
        ])
        examples_text = f"\n\nExamples:\n{examples_text}\n"

    prompt = (
        f"Classify this professional contact into domain tags from: {DOMAIN_TAGS}.\n"
        f"{examples_text}"
        f"Company: {node_data.get('employer', 'unknown')}\n"
        f"Position: {node_data.get('position', 'unknown')}\n"
        f"Return only a JSON list of matching tags, e.g. [\"biotech\", \"investor\"]."
    )

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=100,
        messages=[{"role": "user", "content": prompt}],
    )
    text = response.content[0].text.strip()
    cost = (response.usage.input_tokens * 0.00000025 + response.usage.output_tokens * 0.00000125)

    try:
        tags = json.loads(text)
    except json.JSONDecodeError:
        tags = ["other"]
    return tags, cost

variants = {
    "keyword_only":          {"use_api": False, "few_shot": False, "brave": False},
    "claude_zero_shot":      {"use_api": True,  "few_shot": False, "brave": False},
    "claude_few_shot":       {"use_api": True,  "few_shot": True,  "brave": False},
    "claude_few_shot_brave": {"use_api": True,  "few_shot": True,  "brave": True},
}

In [ ]:
all_tags = sorted(set(t for item in ground_truth for t in item["domain_tags"]))

def tags_to_vector(tags: list) -> list:
    return [1 if t in tags else 0 for t in all_tags]

y_true = [tags_to_vector(item["domain_tags"]) for item in ground_truth]

results = []
for variant_name, params in variants.items():
    with mlflow.start_run(run_name=f"{EXPERIMENT_ID}_{variant_name}"):
        mlflow.log_params(params)

        predictions = []
        total_cost = 0.0

        for item in ground_truth:
            node_data = dict(G.nodes.get(item["node_id"], {}))
            if not params["use_api"]:
                pred_tags = classify_keyword_only(node_data)
                cost = 0.0
            else:
                pred_tags, cost = classify_claude(
                    node_data, few_shot=params["few_shot"]
                )
            total_cost += cost
            predictions.append(tags_to_vector(pred_tags))

        f1 = f1_score(y_true, predictions, average="macro", zero_division=0)
        prec = precision_score(y_true, predictions, average="macro", zero_division=0)
        rec = recall_score(y_true, predictions, average="macro", zero_division=0)
        cost_per_node = total_cost / len(ground_truth)

        mlflow.log_metrics({
            "f1_macro": f1, "precision_macro": prec,
            "recall_macro": rec, "cost_per_node_usd": cost_per_node,
        })
        results.append({"variant": variant_name, "f1": f1, "cost_per_node": cost_per_node})
        print(f"{variant_name}: F1={f1:.4f}, cost/node=${cost_per_node:.5f}")

results_df = pd.DataFrame(results).sort_values("f1", ascending=False)
print(results_df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].barh(results_df["variant"], results_df["f1"])
axes[0].set_xlabel("F1 (macro)")
axes[0].set_title(f"{EXPERIMENT_ID}: Classification Quality")
axes[0].invert_yaxis()

axes[1].barh(results_df["variant"], results_df["cost_per_node"])
axes[1].set_xlabel("Cost per node (USD)")
axes[1].set_title(f"{EXPERIMENT_ID}: API Cost")
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig("experiments/EXP002_domain_classifier/results.png", dpi=150)
plt.show()

## Result and Decision

**Winner:** [fill in after running]
**F1 margin over keyword_only:** [fill in]
**Cost per node:** [fill in]
**Decision:** [VALIDATED | INCONCLUSIVE | REJECTED]

**Decision rule:** If claude_zero_shot F1 < keyword_only F1 + 0.05 → keyword_only wins regardless of absolute scores.

**If VALIDATED — production change:**
File: `netweave/src/taxonomy.py`
Function: `classify_node()`
Change: Replace or augment keyword classifier with winning Claude variant.
Notebook citation: `# Validated in EXP002 — see netweave-lab/experiments/EXP002_domain_classifier/`